# Run Service Notebook
This notebook starts the BentoML service from the trained model and verifies that the API is reachable.

## 1. Start the BentoML service
Run this cell once. It starts the service in a background subprocess and checks `/ping`.

In [ ]:
import os
import sys
import time
import requests
import subprocess
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

service_command = [
    sys.executable,
    '-m',
    'bentoml',
    'serve',
    'seattle_energy.service:EnergyService',
    '--port',
    '3000'
]

env = os.environ.copy()
src_path = str((project_root / 'src').resolve())
env['PYTHONPATH'] = src_path + os.pathsep + env.get('PYTHONPATH', '')

service_proc = subprocess.Popen(
    service_command,
    cwd=str(project_root),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

ping_url = 'http://localhost:3000/ping'
ready = False
for _ in range(20):
    if service_proc.poll() is not None:
        break
    try:
        r = requests.get(ping_url, timeout=1)
        if r.status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)

if ready:
    print('Service started and reachable at', ping_url)
else:
    print('Service did not become ready. Check model availability and dependencies.')
    if service_proc.poll() is not None:
        err_tail = service_proc.stderr.read()[-2000:]
        print('Process exited. Stderr tail:', err_tail)

## 2. Stop the service
Run this when you are done testing.

In [ ]:
if 'service_proc' in globals() and service_proc.poll() is None:
    service_proc.terminate()
    service_proc.wait(timeout=10)
    print('Service stopped.')
else:
    print('No running service process found in this notebook session.')